In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import zipfile

# ---- Project paths (everything lives under this Drive folder) ----
PROJECT_ROOT = "/content/drive/MyDrive/Reinforcement_Learning_Research"
zip_path     = f"{PROJECT_ROOT}/FiFAR.zip"
extract_path = f"{PROJECT_ROOT}/FiFAR_extracted"
DRIVE_OUT    = f"{PROJECT_ROOT}/fifar_prepared"

os.makedirs(extract_path, exist_ok=True)
os.makedirs(DRIVE_OUT, exist_ok=True)

# Extract FiFAR.zip into FiFAR_extracted (skip if already extracted)
if not os.listdir(extract_path):
    print("Extracting FiFAR.zip ...")
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(extract_path)
    print("Extraction complete.")
else:
    print("FiFAR_extracted already has content, skipping extraction.")

# Auto-detect the real base path (handles the extra nested FiFAR/ folder)
candidates = [extract_path, os.path.join(extract_path, "FiFAR")]
BASE_PATH = None
for c in candidates:
    if os.path.exists(os.path.join(c, "alert_data", "Base.csv")):
        BASE_PATH = c
        break

if BASE_PATH is None:
    # fallback: search for Base.csv anywhere under extract_path
    for root, dirs, files in os.walk(extract_path):
        if "Base.csv" in files:
            BASE_PATH = os.path.dirname(os.path.dirname(root)) if os.path.basename(root) == "alert_data" else root
            break

print("BASE_PATH detected as:", BASE_PATH)

# Full structure listing (no file-count truncation this time)
for root, dirs, files in os.walk(BASE_PATH):
    level = root.replace(BASE_PATH, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        print(f"{indent}  {f}")


Mounted at /content/drive
FiFAR_extracted already has content, skipping extraction.
BASE_PATH detected as: /content/drive/MyDrive/Reinforcement_Learning_Research/FiFAR_extracted/FiFAR
FiFAR/
  alert_data/
    Base.csv
    processed_data/
      alerts.parquet
      BAF_alert_model_score.parquet
  testbed/
    test/
      testsize#team_1-var_3/
        batches.csv
        capacity.csv
      testsize#team_3-var_4/
        batches.csv
        capacity.csv
      testsize#team_2-var_3/
        batches.csv
        capacity.csv
      testsize#team_4-var_3/
        batches.csv
        capacity.csv
      testsize#team_2-var_1/
        batches.csv
        capacity.csv
      testsize#team_3-var_1/
        batches.csv
        capacity.csv
      testsize#team_5-hom/
        batches.csv
        capacity.csv
      testsize#team_5-var_2/
        batches.csv
        capacity.csv
      testsize#team_3-hom/
        batches.csv
        capacity.csv
      testsize#team_2-var_4/
        batches.csv
        c

In [2]:
import pandas as pd

BASE_PATH = f"{PROJECT_ROOT}/FiFAR_extracted/FiFAR"

base = pd.read_csv(f"{BASE_PATH}/alert_data/Base.csv")
alerts = pd.read_parquet(f"{BASE_PATH}/alert_data/processed_data/alerts.parquet")
scores = pd.read_parquet(f"{BASE_PATH}/alert_data/processed_data/BAF_alert_model_score.parquet")
expert_preds = pd.read_parquet(f"{BASE_PATH}/synthetic_experts/expert_predictions.parquet")

for name, df in [("base", base), ("alerts", alerts), ("scores", scores), ("expert_preds", expert_preds)]:
    print(f"\n{'='*50}\n{name} | shape={df.shape}\n{'='*50}")
    print(df.columns.tolist())
    print(df.head(2))



base | shape=(1000000, 32)
['fraud_bool', 'income', 'name_email_similarity', 'prev_address_months_count', 'current_address_months_count', 'customer_age', 'days_since_request', 'intended_balcon_amount', 'payment_type', 'zip_count_4w', 'velocity_6h', 'velocity_24h', 'velocity_4w', 'bank_branch_count_8w', 'date_of_birth_distinct_emails_4w', 'employment_status', 'credit_risk_score', 'email_is_free', 'housing_status', 'phone_home_valid', 'phone_mobile_valid', 'bank_months_count', 'has_other_cards', 'proposed_credit_limit', 'foreign_request', 'source', 'session_length_in_minutes', 'device_os', 'keep_alive_session', 'device_distinct_emails_8w', 'device_fraud_count', 'month']
   fraud_bool  income  name_email_similarity  prev_address_months_count  \
0           1     0.9               0.166828                         -1   
1           1     0.9               0.296286                         -1   

   current_address_months_count  customer_age  days_since_request  \
0                          

In [3]:
merged = alerts.join(expert_preds, how="left")
merged = merged.reset_index()

print("Merged shape:", merged.shape)
print("Total nulls:", merged.isnull().sum().sum())
print(merged.columns.tolist())
display(merged.head())

merged.to_parquet(f"{DRIVE_OUT}/merged_fifar_baf.parquet", index=False)
print("Saved to Drive.")
merged.to_csv(f"{DRIVE_OUT}/merged_fifar_baf.csv", index=False)
print("Saved as CSV to Drive.")


Merged shape: (30622, 84)
Total nulls: 0
['case_id', 'fraud_bool', 'income', 'name_email_similarity', 'prev_address_months_count', 'current_address_months_count', 'customer_age', 'days_since_request', 'intended_balcon_amount', 'payment_type', 'zip_count_4w', 'velocity_6h', 'velocity_24h', 'velocity_4w', 'bank_branch_count_8w', 'date_of_birth_distinct_emails_4w', 'employment_status', 'credit_risk_score', 'email_is_free', 'housing_status', 'phone_home_valid', 'phone_mobile_valid', 'bank_months_count', 'has_other_cards', 'proposed_credit_limit', 'foreign_request', 'source', 'session_length_in_minutes', 'device_os', 'keep_alive_session', 'device_distinct_emails_8w', 'device_fraud_count', 'model_score', 'month', 'standard#0', 'standard#1', 'standard#2', 'standard#3', 'standard#4', 'standard#5', 'standard#6', 'standard#7', 'standard#8', 'standard#9', 'standard#10', 'standard#11', 'standard#12', 'standard#13', 'standard#14', 'standard#15', 'standard#16', 'standard#17', 'standard#18', 'standar

,case_id,fraud_bool,income,name_email_similarity,prev_address_months_count,current_address_months_count,customer_age,days_since_request,intended_balcon_amount,payment_type,...,standard#40,standard#41,standard#42,standard#43,standard#44,standard#45,standard#46,standard#47,standard#48,standard#49
0,397063,0,0.7,0.210460,-1,378,30,0.011557,-1.424536,AC,...,0,0,0,0,0,0,0,0,0,0
1,397072,0,0.6,0.608326,-1,65,50,0.003009,-1.070189,AC,...,0,1,1,1,0,1,1,0,0,1
2,397101,0,0.9,0.864950,-1,78,60,0.011739,-1.019647,AC,...,1,1,1,1,0,1,1,1,1,1
3,397126,0,0.1,0.766006,-1,180,60,0.027517,-0.869533,AB,...,0,1,1,0,0,1,1,0,0,0
4,397140,0,0.5,0.192392,-1,70,50,0.001219,15.699609,AA,...,0,0,0,0,0,0,0,0,0,0


Saved to Drive.
Saved as CSV to Drive.


In [4]:
import pandas as pd
import yaml

expert_params = pd.read_parquet(f"{BASE_PATH}/synthetic_experts/expert_parameters.parquet")
prob_of_error = pd.read_parquet(f"{BASE_PATH}/synthetic_experts/prob_of_error.parquet")

with open(f"{BASE_PATH}/synthetic_experts/expert_ids.yaml", "r") as f:
    expert_ids = yaml.safe_load(f)

print("expert_parameters:", expert_params.shape)
print(expert_params.columns.tolist())
print(expert_params.head())
print()
print("prob_of_error:", prob_of_error.shape)
print(prob_of_error.columns.tolist() if hasattr(prob_of_error, 'columns') else type(prob_of_error))
print()
print("expert_ids.yaml keys:", list(expert_ids.keys()) if isinstance(expert_ids, dict) else expert_ids)


expert_parameters: (50, 34)
['income', 'name_email_similarity', 'prev_address_months_count', 'current_address_months_count', 'customer_age', 'days_since_request', 'intended_balcon_amount', 'payment_type', 'zip_count_4w', 'velocity_6h', 'velocity_24h', 'velocity_4w', 'bank_branch_count_8w', 'date_of_birth_distinct_emails_4w', 'employment_status', 'credit_risk_score', 'email_is_free', 'housing_status', 'phone_home_valid', 'phone_mobile_valid', 'bank_months_count', 'has_other_cards', 'proposed_credit_limit', 'foreign_request', 'source', 'session_length_in_minutes', 'device_os', 'keep_alive_session', 'device_distinct_emails_8w', 'device_fraud_count', 'model_score', 'fp_beta', 'fn_beta', 'alpha']
              income  name_email_similarity  prev_address_months_count  \
standard#0  0.288259              -0.407859                   0.041148   
standard#1  0.346841              -0.524905                   0.001898   
standard#2  0.380362              -0.397881                   0.050004   
sta

In [5]:
import os

def build_testbed_manifest(base_path, split_name):
    """split_name = 'train_alert' or 'test'"""
    root = f"{base_path}/testbed/{split_name}"
    manifest = []
    for scenario in os.listdir(root):
        scenario_path = os.path.join(root, scenario)
        if os.path.isdir(scenario_path):
            files = os.listdir(scenario_path)
            manifest.append({
                "scenario": scenario,
                "path": scenario_path,
                "has_batches": "batches.csv" in files,
                "has_capacity": "capacity.csv" in files,
                "has_train_parquet": "train.parquet" in files,
            })
    return pd.DataFrame(manifest)

train_manifest = build_testbed_manifest(BASE_PATH, "train_alert")
test_manifest = build_testbed_manifest(BASE_PATH, "test")

print("Train scenarios:", train_manifest.shape[0])
print("Test scenarios:", test_manifest.shape[0])
train_manifest.head()


Train scenarios: 25
Test scenarios: 25


,scenario,path,has_batches,has_capacity,has_train_parquet
0,shuffle_1#team_5,/content/drive/MyDrive/Reinforcement_Learning_...,True,True,True
1,shuffle_1#team_4,/content/drive/MyDrive/Reinforcement_Learning_...,True,True,True
2,shuffle_2#team_4,/content/drive/MyDrive/Reinforcement_Learning_...,True,True,True
3,shuffle_1#team_1,/content/drive/MyDrive/Reinforcement_Learning_...,True,True,True
4,shuffle_4#team_4,/content/drive/MyDrive/Reinforcement_Learning_...,True,True,True


In [6]:
os.makedirs(DRIVE_OUT, exist_ok=True)

expert_params.to_parquet(f"{DRIVE_OUT}/expert_parameters.parquet")
prob_of_error.to_parquet(f"{DRIVE_OUT}/prob_of_error.parquet")

with open(f"{DRIVE_OUT}/expert_ids.yaml", "w") as f:
    yaml.dump(expert_ids, f)

train_manifest.to_csv(f"{DRIVE_OUT}/train_scenario_manifest.csv", index=False)
test_manifest.to_csv(f"{DRIVE_OUT}/test_scenario_manifest.csv", index=False)

print("All future-need artifacts saved to:", DRIVE_OUT)


All future-need artifacts saved to: /content/drive/MyDrive/Reinforcement_Learning_Research/fifar_prepared


In [7]:
import pandas as pd
import numpy as np

df = pd.read_parquet(f"{DRIVE_OUT}/merged_fifar_baf.parquet")

# ---- Step 1: Missing-value handling (-1 encoded missingness) ----
missing_flag_cols = ['prev_address_months_count', 'intended_balcon_amount',
                      'bank_months_count', 'session_length_in_minutes',
                      'device_distinct_emails_8w']  # verify against actual -1 presence below

for col in df.columns:
    if (df[col] == -1).any():
        print(f"{col}: {(df[col] == -1).sum()} rows with -1")


prev_address_months_count: 29187 rows with -1
current_address_months_count: 23 rows with -1
credit_risk_score: 2 rows with -1
bank_months_count: 12885 rows with -1
session_length_in_minutes: 150 rows with -1
device_distinct_emails_8w: 22 rows with -1


In [8]:
# ---- Step 2: Add missing-value indicator flags (based on actual -1 columns found) ----
missing_flag_cols = [
    'prev_address_months_count',      # 29,187 missing (~95%!) — very sparse, keep as strong signal
    'current_address_months_count',   # 23 missing
    'credit_risk_score',              # 2 missing
    'bank_months_count',              # 12,885 missing (~42%)
    'session_length_in_minutes',      # 150 missing
    'device_distinct_emails_8w',      # 22 missing
]

df_processed = df.copy()

for col in missing_flag_cols:
    df_processed[f'{col}_is_missing'] = (df_processed[col] == -1).astype(int)

print("Missing-flag columns added:", [f'{c}_is_missing' for c in missing_flag_cols])
print(df_processed[[f'{c}_is_missing' for c in missing_flag_cols]].sum())


Missing-flag columns added: ['prev_address_months_count_is_missing', 'current_address_months_count_is_missing', 'credit_risk_score_is_missing', 'bank_months_count_is_missing', 'session_length_in_minutes_is_missing', 'device_distinct_emails_8w_is_missing']
prev_address_months_count_is_missing       29187
current_address_months_count_is_missing       23
credit_risk_score_is_missing                   2
bank_months_count_is_missing               12885
session_length_in_minutes_is_missing         150
device_distinct_emails_8w_is_missing          22
dtype: int64


In [9]:
# ---- Step 3: Categorical encoding ----
categorical_cols = ['payment_type', 'employment_status', 'housing_status', 'source', 'device_os']

print("Unique values per categorical column:")
for col in categorical_cols:
    print(f"{col}: {df_processed[col].unique()}")

df_processed = pd.get_dummies(df_processed, columns=categorical_cols, drop_first=False)
print("\nShape after encoding:", df_processed.shape)


Unique values per categorical column:
payment_type: ['AC', 'AB', 'AA', 'AD', 'AE']
Categories (5, object): ['AA', 'AB', 'AC', 'AD', 'AE']
employment_status: ['CB', 'CA', 'CC', 'CE', 'CD', 'CF', 'CG']
Categories (7, object): ['CA', 'CB', 'CC', 'CD', 'CE', 'CF', 'CG']
housing_status: ['BA', 'BC', 'BB', 'BD', 'BE', 'BF', 'BG']
Categories (7, object): ['BA', 'BB', 'BC', 'BD', 'BE', 'BF', 'BG']
source: ['INTERNET', 'TELEAPP']
Categories (2, object): ['INTERNET', 'TELEAPP']
device_os: ['windows', 'linux', 'macintosh', 'other', 'x11']
Categories (5, object): ['linux', 'macintosh', 'other', 'windows', 'x11']

Shape after encoding: (30622, 111)


In [10]:
# ---- Step 4: Temporal split (based on actual months present: 3,4,5,6,7) ----
print("Month distribution:")
print(df_processed['month'].value_counts().sort_index())

train_df = df_processed[df_processed['month'].isin([3, 4, 5])].copy()
eval_df  = df_processed[df_processed['month'].isin([6, 7])].copy()

print(f"\nTrain (months 3-5): {train_df.shape[0]} rows")
print(f"Eval  (months 6-7): {eval_df.shape[0]} rows")

print("\nFraud rate in train:", train_df['fraud_bool'].mean())
print("Fraud rate in eval:", eval_df['fraud_bool'].mean())


Month distribution:
month
3    8282
4    7612
5    4871
6    5400
7    4457
Name: count, dtype: int64

Train (months 3-5): 20765 rows
Eval  (months 6-7): 9857 rows

Fraud rate in train: 0.11081146159402841
Fraud rate in eval: 0.14334990362179162


In [11]:
# ---- Step 5: Feature scaling (numeric columns only, fit on TRAIN, apply to both) ----
from sklearn.preprocessing import StandardScaler

exclude_from_scaling = ['case_id', 'fraud_bool', 'month'] + \
                        [c for c in train_df.columns if c.startswith('standard#')] + \
                        [c for c in train_df.columns if c.endswith('_is_missing')] + \
                        [c for c in train_df.columns if c.startswith(('payment_type_', 'employment_status_',
                                                                        'housing_status_', 'source_', 'device_os_'))]

numeric_cols = [c for c in train_df.columns if c not in exclude_from_scaling]
print(f"Scaling {len(numeric_cols)} numeric columns")

scaler = StandardScaler()
train_df[numeric_cols] = scaler.fit_transform(train_df[numeric_cols])
eval_df[numeric_cols] = scaler.transform(eval_df[numeric_cols])

print("Scaling done. Sample stats (train):")
print(train_df[numeric_cols[:5]].describe().loc[['mean', 'std']])


Scaling 26 numeric columns
Scaling done. Sample stats (train):
            income  name_email_similarity  prev_address_months_count  \
mean -9.854867e-17          -3.695575e-17              -2.737463e-17   
std   1.000024e+00           1.000024e+00               1.000024e+00   

      current_address_months_count  customer_age  
mean                  1.094985e-16  9.033628e-17  
std                   1.000024e+00  1.000024e+00  


In [12]:
# ---- Step 6: Save final processed splits ----
train_df.to_parquet(f"{DRIVE_OUT}/train_months3to5_processed.parquet", index=False)
eval_df.to_parquet(f"{DRIVE_OUT}/eval_months6to7_processed.parquet", index=False)

import joblib
joblib.dump(scaler, f"{DRIVE_OUT}/scaler.joblib")

print("Final processed train/eval + scaler saved to:", DRIVE_OUT)
print("Train shape:", train_df.shape)
print("Eval shape:", eval_df.shape)


Final processed train/eval + scaler saved to: /content/drive/MyDrive/Reinforcement_Learning_Research/fifar_prepared
Train shape: (20765, 111)
Eval shape: (9857, 111)
